# 04 — Ensemble Tuning & Threshold Selection

Tune the ensemble weights, calibrate the combined score, and select
optimal decision thresholds for the accept/review/hold pipeline.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from config.settings import MODELS_DIR, SPLITS_DIR
from src.ensemble.scorer import EnsembleScorer, weighted_score, make_decision
from src.ensemble.calibration import (
    create_calibrator,
    evaluate_calibration,
    fit_and_save,
)
from evaluation.metrics import compute_all_metrics, print_report

sns.set_theme(style='whitegrid')
%matplotlib inline

In [ ]:
# Load validation and test data
val_df = pd.read_parquet(SPLITS_DIR / 'val.parquet')
test_df = pd.read_parquet(SPLITS_DIR / 'test.parquet')
print(f'Val: {len(val_df)}, Test: {len(test_df)}')

In [ ]:
# NOTE: Replace these with actual model predictions.
# This cell simulates component scores for demonstration.
np.random.seed(42)
n_val = len(val_df)
n_test = len(test_df)

# Simulate scores (replace with real predictions)
val_scores = {
    'statistical': np.random.beta(2, 5, n_val) + val_df['label'].values * 0.3,
    'codebert': np.random.beta(2, 5, n_val) + val_df['label'].values * 0.4,
}
test_scores = {
    'statistical': np.random.beta(2, 5, n_test) + test_df['label'].values * 0.3,
    'codebert': np.random.beta(2, 5, n_test) + test_df['label'].values * 0.4,
}

# Clip to [0, 1]
for key in val_scores:
    val_scores[key] = np.clip(val_scores[key], 0, 1)
    test_scores[key] = np.clip(test_scores[key], 0, 1)

In [ ]:
# Fit ensemble weights on validation set
scorer = EnsembleScorer()
val_component_list = [
    {k: float(v[i]) for k, v in val_scores.items()}
    for i in range(n_val)
]
scorer.fit_weights(val_component_list, val_df['label'].tolist())
print('Ensemble weights learned.')

In [ ]:
# Compute ensemble scores on test set
test_ensemble_scores = []
for i in range(n_test):
    cs = {k: float(v[i]) for k, v in test_scores.items()}
    result = scorer.score(cs)
    test_ensemble_scores.append(result['risk_score'])

test_ensemble_scores = np.array(test_ensemble_scores)
print('Ensemble test scores computed.')
print_report(test_df['label'].values, test_ensemble_scores)

In [ ]:
# Calibrate the ensemble scores
val_ensemble = []
for i in range(n_val):
    cs = {k: float(v[i]) for k, v in val_scores.items()}
    result = scorer.score(cs)
    val_ensemble.append(result['risk_score'])

val_ensemble = np.array(val_ensemble)

calibrator = fit_and_save(val_ensemble, val_df['label'].values, method='platt')
calibrated_test = calibrator.calibrate_batch(test_ensemble_scores)

print('\nBefore calibration:')
print(evaluate_calibration(test_ensemble_scores, test_df['label'].values))
print('\nAfter calibration:')
print(evaluate_calibration(calibrated_test, test_df['label'].values))

In [ ]:
# Threshold sweep
thresholds = np.arange(0.1, 0.9, 0.05)
results = []
for t in thresholds:
    m = compute_all_metrics(test_df['label'].values, calibrated_test, threshold=t)
    results.append({'threshold': t, 'f1': m['f1'], 'precision': m['precision'], 'recall': m['recall']})

results_df = pd.DataFrame(results)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(results_df['threshold'], results_df['f1'], 'b-o', label='F1')
ax.plot(results_df['threshold'], results_df['precision'], 'g--', label='Precision')
ax.plot(results_df['threshold'], results_df['recall'], 'r--', label='Recall')
ax.set_xlabel('Threshold')
ax.set_ylabel('Score')
ax.set_title('Threshold Tuning')
ax.legend()
ax.grid(True, alpha=0.3)

# Mark the accept/review/hold zones
ax.axvline(x=0.4, color='orange', linestyle=':', alpha=0.7, label='Accept/Review boundary')
ax.axvline(x=0.7, color='red', linestyle=':', alpha=0.7, label='Review/Hold boundary')
ax.legend()
plt.tight_layout()
plt.show()

best = results_df.loc[results_df['f1'].idxmax()]
print(f"\nBest F1={best['f1']:.3f} at threshold={best['threshold']:.2f}")

In [ ]:
# Decision distribution
decisions = [make_decision(s) for s in calibrated_test]
dec_series = pd.Series(decisions)
print('\nDecision distribution:')
print(dec_series.value_counts())

fig, ax = plt.subplots(figsize=(8, 5))
dec_series.value_counts().plot.bar(ax=ax, color=['green', 'orange', 'red'])
ax.set_title('Decision Distribution on Test Set')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Save final scorer
scorer.calibrator = calibrator
scorer.save()
print('Ensemble scorer saved.')